# [![](https://img.shields.io/badge/Thiết%20kế-CookApps.net-0075ff)](https://cookapps.net) [![](https://img.shields.io/badge/Hướng%20dẫn-SD%20Nhập%20Môn-5858e6)](#) [![](https://img.shields.io/badge/Group-Support-0075ff)](#) [![](https://img.shields.io/badge/Mở-AUTOMATIC1111%20WebUI-blue)](https://colab.research.google.com/github/25410166/Comfy_Collab/blob/master/colab/Automatic1111_Studio.ipynb)

*`💡 Công Cụ Download Model Chuyên Dụng (CivitAI & HuggingFace Tốc Độ Cao) - CookApps.net`*

### ⚡ Tính năng chuyên dụng:
- 🚀 **Tải đa luồng `aria2c`**: Tối đa băng thông, tự động nhận diện file.
- 🔑 **Bypass 403 Forbidden Civitai**: Tự động nhận diện `modelVersionId` và nạp API Token để tải model riêng tư hoặc giới hạn.
- 🏷️ **Đổi tên file trực tiếp**: Hỗ trợ cú pháp `@=tên_muốn_đổi` (ví dụ: `https://civitai.com/models/...@=my_lora`).
- 💾 **Đồng bộ Google Drive**: Lưu trực tiếp vào `MyDrive/ComfyStudio/models` (dùng chung ngay lập tức cho WebUI).
- 🛑 **Chống tải trùng thông minh 100%**: Bỏ qua các file đã có trên Drive để bảo vệ dung lượng.

In [ ]:
#@title 1. Kết nối Google Drive & Khởi tạo Thư mục Models
import os
from google.colab import drive

print("🚀 Đang kết nối Google Drive...")
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
FOLDERS = [
    "checkpoints",
    "loras",
    "vae",
    "controlnet",
    "upscale_models",
    "embeddings"
]

for f in FOLDERS:
    os.makedirs(os.path.join(DRIVE_BASE, f), exist_ok=True)

# Cài đặt aria2 tốc độ cao
!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

print(f"✅ Đã kết nối và sẵn sàng thư mục Models trên Drive: {DRIVE_BASE}")

In [ ]:
#@title 📦 2. Tải Nhanh Gói Model Tuyển Chọn (Fast / Full)
#@markdown Chọn gói cần tải vào Google Drive:
#@markdown - ⚡ **Fast**: Tải bộ model cốt lõi tối thiểu để chạy ngay (SDXL Turbo + SD1.5 Realistic + VAE + Upscaler + 1 LoRA).
#@markdown - 📦 **Full**: Tải trọn bộ tất cả các model tuyển chọn còn thiếu trong kho.

Che_Do = "⚡ Fast" #@param ["⚡ Fast", "📦 Full", "🔍 Kiểm tra trạng thái trên Google Drive"]

import os
import sys
import subprocess
from huggingface_hub import hf_hub_download

# --- BẢO MẬT & MÃ HÓA TOKEN (CHẠY NGẦM) ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open('/content/.vault_tokens.json', 'w') as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = '', pwd: str = '') -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists('/content/.vault_tokens.json'):
            cached = json.load(open('/content/.vault_tokens.json'))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ''

Civitai_API_Token = get_vault_token('civitai', '', Mat_Khau_Mo_Khoa)
HuggingFace_Token = get_vault_token('huggingface', '', Mat_Khau_Mo_Khoa)

TARGET_BASE = '/content/drive/MyDrive/ComfyStudio/models'

CURATED_MODELS = [
    {
        'id': 'krea2_turbo',
        'is_fast': True,
        'name': 'Core Turbo Checkpoint [SDXL]',
        'category': 'checkpoints',
        'filename': 'lustify_workflows_krea_2_sdxl.safetensors',
        'aliases': ['lustify', 'krea2', 'krea_2', 'lustifynsfwcheckpoint'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'url': 'https://civitai.com/api/download/models/3112728'
    },
    {
        'id': 'realistic_vision_v6',
        'is_fast': True,
        'name': 'Core Photorealistic Checkpoint [SD1.5]',
        'category': 'checkpoints',
        'filename': 'realisticVisionV60B1_v51VAE.safetensors',
        'aliases': ['realisticvision', 'realistic_vision', 'v51hypervae', 'v51vae'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'url': 'https://civitai.com/api/download/models/130072?type=Model&format=SafeTensor&size=pruned&fp=fp16',
        'alt_hf': {'repo_id': 'SG161222/Realistic_Vision_V5.1_noVAE', 'filename': 'Realistic_Vision_V5.1_fp16-no-ema.safetensors'}
    },
    {
        'id': 'wai_illustrious',
        'is_fast': False,
        'name': 'Anime Illustrious Checkpoint [SDXL]',
        'category': 'checkpoints',
        'filename': 'waiIllustriousSDXL_v170.safetensors',
        'aliases': ['waiillustrious', 'wai-illustrious', 'illustrious_sdxl'],
        'min_size_mb': 2000,
        'source_type': 'civitai',
        'url': 'https://civitai.com/api/download/models/2883731'
    },
    {
        'id': 'majicmix_realistic',
        'is_fast': False,
        'name': 'Asian Portrait Checkpoint [SD1.5]',
        'category': 'checkpoints',
        'filename': 'majicmixRealistic_v7.safetensors',
        'aliases': ['majicmix', 'majicmixrealistic'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'url': 'https://civitai.com/api/download/models/176425'
    },
    {
        'id': 'pony_diffusion_v6',
        'is_fast': False,
        'name': 'Pony Diffusion V6 Checkpoint [SDXL]',
        'category': 'checkpoints',
        'filename': 'ponyDiffusionV6XL_v6StartWithThisOne.safetensors',
        'aliases': ['ponydiffusion', 'pony_v6', 'ponydiffusionv6xl'],
        'min_size_mb': 2000,
        'source_type': 'civitai',
        'url': 'https://civitai.com/api/download/models/290640'
    },
    {
        'id': 'lora_krea_2_nsfw',
        'is_fast': True,
        'name': 'Detail Enhancer LoRA',
        'category': 'loras',
        'filename': 'krea_2_nsfw_v4.safetensors',
        'aliases': ['knp_000003000', 'krea_2_nsfw', 'knpv4'],
        'min_size_mb': 10,
        'source_type': 'civitai',
        'url': 'https://civitai.com/api/download/models/3147117'
    },
    {
        'id': 'lora_expressive_h',
        'is_fast': False,
        'name': 'Expressive Style LoRA',
        'category': 'loras',
        'filename': 'Expressive_H-000001.safetensors',
        'aliases': ['expressive_h', 'expressiveh'],
        'min_size_mb': 10,
        'source_type': 'civitai',
        'url': 'https://civitai.com/api/download/models/382152'
    },
    {
        'id': 'lora_breasts_out',
        'is_fast': False,
        'name': 'Fashion Clothing LoRA',
        'category': 'loras',
        'filename': '2BoutOval_K2v3_400.safetensors',
        'aliases': ['2boutoval', 'breasts_out', 'breastsout'],
        'min_size_mb': 5,
        'source_type': 'civitai',
        'url': 'https://civitai.com/api/download/models/3134691'
    },
    {
        'id': 'vae_sdxl',
        'is_fast': True,
        'name': 'SDXL Color Balance VAE',
        'category': 'vae',
        'filename': 'sdxl_vae.safetensors',
        'aliases': ['sdxl_vae', 'sdxlvae'],
        'min_size_mb': 50,
        'source_type': 'hf',
        'repo_id': 'stabilityai/sdxl-vae',
        'hf_filename': 'sdxl_vae.safetensors'
    },
    {
        'id': 'upscaler_4x_ultrasharp',
        'is_fast': True,
        'name': '4K Upscaler UltraSharp',
        'category': 'upscale_models',
        'filename': '4x-UltraSharp.pth',
        'aliases': ['4x-ultrasharp', '4x_ultrasharp', 'ultrasharp'],
        'min_size_mb': 10,
        'source_type': 'hf',
        'repo_id': 'lokcx/4x-Ultrasharp',
        'hf_filename': '4x-UltraSharp.pth'
    }
]

def find_existing_model(target_base, category, primary_name, aliases=None, min_mb=10):
    folder = os.path.join(target_base, category)
    if not os.path.exists(folder):
        return None, 0
    candidates = [primary_name.lower()]
    if aliases:
        for a in aliases:
            candidates.append(a.lower())
    min_bytes = min_mb * 1024 * 1024
    try:
        for fname in os.listdir(folder):
            full_path = os.path.join(folder, fname)
            if not os.path.isfile(full_path):
                continue
            if fname.endswith('.aria2'):
                continue
            sz = os.path.getsize(full_path)
            if sz < min_bytes:
                continue
            lower_fname = fname.lower()
            for cand in candidates:
                if cand == lower_fname or cand in lower_fname:
                    return full_path, sz
    except Exception:
        pass
    return None, 0

if Che_Do.startswith('🔍'):
    print('=' * 80)
    print(f'📊 BẢNG QUÉT TRẠNG THÁI MODEL TRÊN GOOGLE DRIVE ({TARGET_BASE})')
    print('=' * 80)
    count = 0
    total_sz = 0
    for idx, item in enumerate(CURATED_MODELS, 1):
        fpath, sz = find_existing_model(TARGET_BASE, item['category'], item['filename'], item.get('aliases'), item['min_size_mb'])
        if fpath:
            count += 1
            total_sz += sz
            sz_str = f"{sz / (1024**3):.2f} GB" if sz >= 1024**3 else f"{sz / (1024**2):.1f} MB"
            status = '✅ ĐÃ CÓ'
        else:
            sz_str = f"~{item['min_size_mb']} MB"
            status = '❌ CHƯA CÓ'
        print(f"[{idx:<2}] {status:<10} | {item['category']:<15} | {sz_str:<10} | {item['name']}")
    print('=' * 80)
    print(f"📈 Tổng kết: Đã có {count}/{len(CURATED_MODELS)} model ({total_sz/(1024**3):.2f} GB). Còn thiếu {len(CURATED_MODELS)-count} model.")
else:
    is_fast = 'Fast' in Che_Do
    targets = [m for m in CURATED_MODELS if (m.get('is_fast') if is_fast else True)]
    mode_label = '⚡ FAST MODE (Bộ model cốt lõi)' if is_fast else '📦 FULL MODE (Toàn bộ model tuyển chọn)'
    print('=' * 80)
    print(f'🚀 BẮT ĐẦU TẢI: {mode_label}')
    print(f'🎯 Lưu vào: {TARGET_BASE}')
    print('=' * 80)
    
    for idx, item in enumerate(targets, 1):
        category = item['category']
        filename = item['filename']
        dest_dir = os.path.join(TARGET_BASE, category)
        os.makedirs(dest_dir, exist_ok=True)
        dest_file = os.path.join(dest_dir, filename)
        
        existing_file, fsize = find_existing_model(TARGET_BASE, category, filename, item.get('aliases'), item['min_size_mb'])
        if existing_file:
            sz = fsize / (1024**3) if fsize >= 1024**3 else fsize / (1024**2)
            unit = 'GB' if fsize >= 1024**3 else 'MB'
            print(f"⏩ [{idx}/{len(targets)}] [BỎ QUA - ĐÃ CÓ] {item['name']} ({sz:.2f} {unit})")
            continue
            
        print(f"\n⬇️ [{idx}/{len(targets)}] [ĐANG TẢI] {item['name']}...")
        if item['source_type'] == 'civitai':
            url = item['url']
            if Civitai_API_Token and 'token=' not in url:
                url += f'?token={Civitai_API_Token}' if '?' not in url else f'&token={Civitai_API_Token}'
            cmd = [
                'aria2c', '-c', '-x', '16', '-s', '16', '-k', '1M',
                '--file-allocation=none',
                '--header=User-Agent: Mozilla/5.0',
                '-d', dest_dir,
                '-o', filename,
                url
            ]
            ret = subprocess.run(cmd)
            if ret.returncode != 0 and 'alt_hf' in item:
                hf = item['alt_hf']
                try:
                    hf_hub_download(repo_id=hf['repo_id'], filename=hf['filename'], local_dir=dest_dir, local_dir_use_symlinks=False)
                    tmp = os.path.join(dest_dir, hf['filename'])
                    if tmp != dest_file and os.path.exists(tmp):
                        os.rename(tmp, dest_file)
                except Exception:
                    pass
        elif item['source_type'] == 'hf':
            try:
                dl_kwargs = {
                    'repo_id': item['repo_id'],
                    'filename': item['hf_filename'],
                    'local_dir': dest_dir,
                    'local_dir_use_symlinks': False
                }
                if HuggingFace_Token:
                    dl_kwargs['token'] = HuggingFace_Token
                dl_path = hf_hub_download(**dl_kwargs)
                if dl_path != dest_file and os.path.exists(dl_path):
                    if os.path.exists(dest_file):
                        os.remove(dest_file)
                    os.rename(dl_path, dest_file)
            except Exception as e:
                print(f"❌ Lỗi: {e}")
                
        existing_file, fsize = find_existing_model(TARGET_BASE, category, filename, item.get('aliases'), item['min_size_mb'])
        if existing_file:
            print(f"   ✅ Tải thành công: {filename}")
        else:
            print(f"   ⚠️ Tải chưa hoàn tất: {filename}")
            
    print('=' * 80)
    print('🎉 Hoàn tất tiến trình tải gói model!')

In [ ]:
#@title 🧨 3. Download Model Trực Tiếp (CivitAI, Huggingface)
#@markdown Dán link model trực tiếp từ Civitai hoặc HuggingFace vào các ô tương ứng:
#@markdown - ⚡ **Bypass 403 Civitai**: Tự động nhận diện `modelVersionId` và nạp API Token để tải tốc độ cao.
#@markdown - 🛑 **Chống tải trùng thông minh**: Tự động kiểm tra trên Google Drive trước khi tải để bảo vệ dung lượng.
#@markdown - 🏷️ **Hỗ trợ đổi tên theo ý muốn**: Thêm `@=tên_muốn_đổi` vào cuối link (ví dụ: `https://civitai.com/models/...@=my_lora`).

Checkpoint__ = "" #@param {type:"string"}
Lora_Link___ = "" #@param {type:"string"}
Controlnet__ = "" #@param {type:"string"}
VAE_Link____ = "" #@param {type:"string"}
Upscaler____ = "" #@param {type:"string"}

# Link tùy chọn khác (chọn thư mục lưu tương ứng)
Custom_Link_ = "" #@param {type:"string"}
Thu_Muc_Luu_ = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "embeddings"]

import os
import re
import sys
import json
import urllib.request
import subprocess

TARGET_BASE = "/content/drive/MyDrive/ComfyStudio/models"

# Tự động lấy token giải mã từ Vault
Civitai_API_Token = get_vault_token("civitai", "", Mat_Khau_Mo_Khoa) if 'get_vault_token' in globals() else ""

def split_string_at(link):
    delimiter = "@="
    if delimiter in link:
        parts = link.split(delimiter)
        name = parts[1].strip()
        if not any(name.endswith(ext) for ext in ['.safetensors', '.ckpt', '.pt', '.pth', '.bin']):
            name = f"{name}.safetensors"
        return parts[0].strip(), name
    return link.strip(), None

def resolve_model_info(url, token=""):
    headers = {"User-Agent": "Mozilla/5.0"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    
    if "huggingface.co" in url and "blob" in url:
        url = url.replace("/blob/", "/resolve/")
    
    if "civitai.com" in url or "civitai.red" in url:
        ver_match = re.search(r'modelVersionId=(\d+)', url)
        if ver_match:
            ver_id = ver_match.group(1)
            api_url = f"https://civitai.com/api/v1/model-versions/{ver_id}"
            try:
                req = urllib.request.Request(api_url, headers=headers)
                with urllib.request.urlopen(req) as resp:
                    data = json.loads(resp.read().decode('utf-8'))
                    files = data.get("files", [])
                    fname = files[0].get("name") if files else f"{ver_id}.safetensors"
                    size_kb = files[0].get("sizeKB", 0) if files else 0
                    dl_url = f"https://civitai.com/api/download/models/{ver_id}"
                    if token:
                        dl_url += f"?token={token}"
                    return fname, dl_url, size_kb / 1024
            except Exception as e:
                return f"{ver_id}.safetensors", f"https://civitai.com/api/download/models/{ver_id}?token={token}", 0
        
        model_match = re.search(r'/models/(\d+)', url)
        if model_match:
            model_id = model_match.group(1)
            api_url = f"https://civitai.com/api/v1/models/{model_id}"
            try:
                req = urllib.request.Request(api_url, headers=headers)
                with urllib.request.urlopen(req) as resp:
                    data = json.loads(resp.read().decode('utf-8'))
                    versions = data.get("modelVersions", [])
                    if versions:
                        ver = versions[0]
                        ver_id = ver.get("id")
                        files = ver.get("files", [])
                        fname = files[0].get("name") if files else f"{ver_id}.safetensors"
                        size_kb = files[0].get("sizeKB", 0) if files else 0
                        dl_url = f"https://civitai.com/api/download/models/{ver_id}"
                        if token:
                            dl_url += f"?token={token}"
                        return fname, dl_url, size_kb / 1024
            except Exception as e:
                return f"{model_id}.safetensors", url, 0
                
    clean_url = url.split("?")[0]
    fname = os.path.basename(clean_url)
    return fname, url, 0

tasks = []
if Checkpoint__.strip():
    tasks.append(('checkpoints', Checkpoint__.strip()))
if Lora_Link___.strip():
    tasks.append(('loras', Lora_Link___.strip()))
if Controlnet__.strip():
    tasks.append(('controlnet', Controlnet__.strip()))
if VAE_Link____.strip():
    tasks.append(('vae', VAE_Link____.strip()))
if Upscaler____.strip():
    tasks.append(('upscale_models', Upscaler____.strip()))
if Custom_Link_.strip():
    tasks.append((Thu_Muc_Luu_, Custom_Link_.strip()))

if not tasks:
    print("ℹ️ Bạn chưa nhập link model nào vào các ô ở trên.")
    print("💡 Hãy dán link Checkpoint, LoRA, ControlNet hoặc Custom Link rồi bấm Play để tải đa luồng!")
else:
    print("=" * 80)
    print(f"🚀 BẮT ĐẦU TẢI {len(tasks)} MODEL VÀO GOOGLE DRIVE: {TARGET_BASE}")
    print("=" * 80)
    
    for folder_cat, raw_link in tasks:
        clean_url, custom_name = split_string_at(raw_link)
        detected_name, download_url, estimated_mb = resolve_model_info(clean_url, Civitai_API_Token)
        final_filename = custom_name if custom_name else detected_name
        if not final_filename:
            final_filename = "downloaded_model.safetensors"
            
        target_dir = os.path.join(TARGET_BASE, folder_cat)
        os.makedirs(target_dir, exist_ok=True)
        dest_path = os.path.join(target_dir, final_filename)
        
        # 1. Kiểm tra chống tải trùng
        if os.path.exists(dest_path) and os.path.getsize(dest_path) > 10 * 1024 * 1024:
            sz_bytes = os.path.getsize(dest_path)
            sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
            print(f"⏩ [BỎ QUA - ĐÃ CÓ SẴN] {final_filename} ({sz_str}) đã có trên Google Drive!")
            continue
            
        # 2. Tải đa luồng aria2c
        print(f"\n⬇️ [ĐANG TẢI] {final_filename}")
        print(f"   URL: {clean_url}")
        print(f"   Lưu vào: {dest_path}")
        
        cmd = [
            "aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M",
            "--file-allocation=none",
            "--header=User-Agent: Mozilla/5.0",
            "-d", target_dir,
            "-o", final_filename,
            download_url
        ]
        subprocess.run(cmd)
        
        # 3. Xác thực sau tải
        if os.path.exists(dest_path) and os.path.getsize(dest_path) > 10 * 1024 * 1024:
            sz_bytes = os.path.getsize(dest_path)
            sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
            print(f"🎉 TẢI THÀNH CÔNG: {final_filename} ({sz_str})")
        else:
            print(f"❌ Tải thất bại hoặc file không hợp lệ: {dest_path}")
            
    print("=" * 80)
    print("✅ Hoàn tất tiến trình tải model tùy chọn!")

In [ ]:
#@title ✨ 4. Custom Download (Tải Link Bất Kỳ)
#@markdown Tải link direct bất kỳ từ HuggingFace, Civitai hoặc máy chủ bên ngoài:
Link = "" #@param {type:"string"}
Thu_Muc = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "embeddings"]
Ten_File = "" #@param {type:"string"}

import os
import subprocess

TARGET_BASE = "/content/drive/MyDrive/ComfyStudio/models"

clean_link = Link.strip()
if not clean_link:
    print("⚠️ Vui lòng dán đường link vào ô [Link] ở trên.")
else:
    target_dir = os.path.join(TARGET_BASE, Thu_Muc)
    os.makedirs(target_dir, exist_ok=True)
    fname = Ten_File.strip() if Ten_File.strip() else os.path.basename(clean_link.split("?")[0])
    dest_path = os.path.join(target_dir, fname)
    
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 5 * 1024 * 1024:
        print(f"⏩ [BỎ QUA - ĐÃ CÓ] File '{fname}' đã có sẵn trên Drive!")
    else:
        print(f"⬇️ Bắt đầu tải {fname} vào {Thu_Muc}...")
        cmd = [
            "aria2c", "--console-log-level=error", "--summary-interval=10",
            "-c", "-x", "16", "-s", "16", "-k", "1M",
            "--file-allocation=none",
            "--header=User-Agent: Mozilla/5.0",
            "-d", target_dir,
            "-o", fname,
            clean_link
        ]
        subprocess.run(cmd)
        if os.path.exists(dest_path):
            print(f"🎉 Tải thành công: {fname}!")
        else:
            print(f"❌ Tải thất bại.")